# A fresh numerical check of the fifth Bessel moment

**All the work was done by AI.** Jonas Matuzas is responsible for the claims.

This notebook independently recomputes both sides of the conjecture in
[the published v4 paper](https://doi.org/10.5281/zenodo.22793737).
It uses new numerical routes rather than the paper's certificate programs or stored answers.
It is an AI-executed calculation, **not independent human peer review**.

**The equality remains conjectural.** Agreement at many digits is numerical evidence, not a proof.

Open in Google Colab, select **Runtime → Run all**, and use the ordinary CPU runtime.
No GPU, mounted Google Drive, private data or access tokens are needed.
The notebook contains its calculation code; only the pinned `mpmath` package may need downloading.
The two default runs use 40 and 60 significant digits, with 20 guard digits internally.
Runtime depends on the hosted CPU. All outputs below must be regenerated to count as your own run.

[GitHub source and method notes](https://github.com/Matuzas77/bessel5-reproducibility/blob/main/docs/INDEPENDENT_CHECK.md)


## 1. Define exactly what is being tested

The measure is **$dt$**, not $t\,dt$:

$$
A_0=\int_0^\infty K_0(t)^5\,dt
\stackrel{?}{=}\frac{\pi}{4}\left[(125-13\sqrt5)\Re(L^2)
+(41\sqrt5-125)\Im(L^2)\right].
$$

Here $L=L(f,2)=4\pi^2\int_0^\infty y f(iy)\,dy$, the classical unshifted value.
The **complex square** is taken before the real and imaginary parts.
With $\eta_d=\eta(d\tau)$, define

$$
P_1=\eta_2\eta_6^2\eta_{10}^4/\eta_{30},\quad
P_2=\eta_2^4\eta_{10}\eta_{30}^2/\eta_6,
$$
$$
P_3=\eta_6^4\eta_{10}^2\eta_{30}/\eta_2,\quad
P_4=\eta_2^2\eta_6\eta_{30}^4/\eta_{10},
$$
$$
\omega=(1-2i)/\sqrt5,\quad
\epsilon=(5+4\sqrt5+i(2\sqrt5-10))/15,
\quad f=P_1+\omega P_2+3\epsilon P_3+3\epsilon\bar\omega P_4.
$$

Freshly generated Fourier coefficients check the embedding
$a_1=1, a_3=\sqrt5-2i, a_5=\sqrt5(1+2i), a_7=-8i$.
No saved integral, L-value, coefficient table or certificate is a numerical input.


## 2. Pin the numerical library and record the runtime

In [1]:
import sys, platform, subprocess, tempfile
from importlib.metadata import version, PackageNotFoundError
try:
    needs_install = version("mpmath") != "1.3.0"
except PackageNotFoundError:
    needs_install = True
if needs_install:
    requirement = "mpmath==1.3.0 --hash=sha256:a0b2b9fe80bbcd81a6647ff13108738cfb482d481d826cc0e02f5b35e5c88d2c\n"
    with tempfile.NamedTemporaryFile(mode="w", suffix=".txt") as handle:
        handle.write(requirement)
        handle.flush()
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet",
                               "--require-hashes", "--only-binary=:all:", "-r", handle.name])
import mpmath
if mpmath.__version__ != "1.3.0":
    raise RuntimeError("Restart the runtime after installing the pinned mpmath version")
print("Python:", sys.version)
print("Platform:", platform.platform())
print("mpmath:", mpmath.__version__)
print("All the work was done by AI.")

Python: 3.12.12 (main, Oct  9 2025, 11:07:00) [Clang 17.0.0 (clang-1700.4.4.1)]
Platform: macOS-27.0-arm64-arm-64bit
mpmath: 1.3.0
All the work was done by AI.


## 3. Compute the Bessel side directly

The code below calls `mpmath.besselk(0, t)` and quadratures it directly.
The logarithmic singularity near zero is handled by a coordinate change.
An alternate run changes that coordinate and the interval partition.
Analytic endpoint-tail bounds are recorded, but adaptive quadrature and rounding
are **not certified interval arithmetic**.


In [2]:
"""Fresh quadrature of A0 = integral_0^infinity K0(t)^5 dt.

All the work was done by AI.

Only the defining Bessel integral and mpmath's special functions enter this
calculation. No archived Bessel values, certificates, modular constants, or
conjectured right-hand side are read. Results are arbitrary-precision numerical
approximations, not interval-certified enclosures or proofs of the conjecture.
"""
from __future__ import annotations

import time
import mpmath


def compute_bessel(dps: int, variant: str = "primary") -> dict:
    """Compute the fifth K0 moment without using the conjectured answer.

    ``primary`` maps 0<t<1 to t=exp(-x), truncating at x=X.
    ``alternate`` maps 0<t<1 to t=u**16, 0<u<1, and changes the upper
    integration partitions. Both use mpmath's tanh-sinh quadrature.

    The returned value has extra display digits for convergence comparisons;
    the requested precision is not an assertion of certified accuracy.
    Endpoint truncation bounds are analytic inequalities evaluated in ordinary
    arbitrary-precision floating point. The quadrature error is heuristic.
    """
    if isinstance(dps, bool) or not isinstance(dps, int) or not 20 <= dps <= 200:
        raise ValueError("dps must be an integer from 20 through 200")
    if variant not in {"primary", "alternate"}:
        raise ValueError("variant must be 'primary' or 'alternate'")

    started = time.perf_counter()
    # A private context avoids changing a notebook's global mpmath precision.
    ctx = mpmath.mp.clone()
    guard = 20
    ctx.dps = dps + guard
    stringify = lambda value: ctx.nstr(value, dps + 8)
    zero = ctx.mpf(0)
    one = ctx.mpf(1)
    low_error = zero

    if variant == "primary":
        X = (dps + 35) * ctx.log(10)
        low_nodes = [ctx.mpf(v) for v in [0, 1, 4, 10, 25, 60] if v < X] + [X]

        def low_integrand(x):
            t = ctx.exp(-x)
            return t * ctx.besselk(0, t) ** 5

        low, low_error = ctx.quad(low_integrand, low_nodes, error=True)
        # For 0<t<=1, z=t**2/4 and the convergent K0/I0 series give
        # K0(t) <= exp(z)*(log(2/t)-EulerGamma+z)
        #        <= exp(1/4)*(1-log(t)).
        # Integrating this fifth power over 0<t<exp(-X) yields:
        y = X + 1
        polynomial = y**5 + 5*y**4 + 20*y**3 + 60*y**2 + 120*y + 120
        lower_tail = ctx.exp(ctx.mpf(5)/4 - X) * polynomial
        low_settings = {
            "substitution": "t = exp(-x)",
            "x_partitions": [stringify(v) for v in low_nodes],
            "small_t_cutoff": stringify(ctx.exp(-X)),
            "omitted_lower_tail_bound_formula":
                "exp(5/4-X)*((X+1)^5+5*(X+1)^4+20*(X+1)^3+60*(X+1)^2+120*(X+1)+120)",
        }
    else:
        low_nodes = [zero, ctx.mpf("0.25"), ctx.mpf("0.5"), ctx.mpf("0.75"), one]

        def low_integrand(u):
            if not u:
                return zero  # The limiting value is zero.
            return 16 * u**15 * ctx.besselk(0, u**16) ** 5

        low, low_error = ctx.quad(low_integrand, low_nodes, error=True)
        lower_tail = zero  # No finite lower-end truncation in this variant.
        low_settings = {
            "substitution": "t = u^16",
            "u_partitions": [stringify(v) for v in low_nodes],
            "small_t_cutoff": "0",
            "omitted_lower_tail_bound_formula":
                "0 (no explicit lower-end truncation; quadrature error remains numerical)",
        }

    # The rigorous real-variable inequality K0(t)<=sqrt(pi/(2*t))*exp(-t)
    # follows from K0(t)=int_0^infinity exp(-t*cosh(u))du and
    # cosh(u)>=1+u^2/2. Consequently the omitted upper tail is at most
    # (pi/(2*T))^(5/2)*exp(-5*T)/5. We evaluate that expression numerically.
    tail_extra = 12 if variant == "primary" else 17
    T = (dps + tail_extra) * ctx.log(10) / 5
    candidates = [1, 2, 5, 10, 15, 20, 30, 45, 65, 90, 120]
    if variant == "alternate":
        candidates = [1, "1.5", 3, 7, 12, 18, 26, 38, 55, 75, 100, 125]
    upper_nodes = [ctx.mpf(v) for v in candidates if ctx.mpf(v) < T] + [T]
    high = zero
    high_error = zero
    segment_details = []
    for a, b in zip(upper_nodes[:-1], upper_nodes[1:]):
        # Scale away exp(-5*a). This permits lower local working precision
        # while preserving an absolute-error budget of approximately dps+20
        # digits. This is a numerical efficiency choice, NOT an error proof.
        local_digits = max(25, dps + guard - int(5*a/ctx.log(10)))
        with ctx.workdps(local_digits):
            normalization = ctx.exp(5*a)
            normalized, error = ctx.quad(
                lambda t: normalization * ctx.besselk(0, t)**5,
                [a, b], error=True,
            )
        scale = ctx.exp(-5*a)
        high += normalized * scale
        high_error += abs(error) * scale
        segment_details.append({
            "interval": [stringify(a), stringify(b)],
            "working_dps": local_digits,
        })

    upper_tail = (ctx.pi/(2*T))**(ctx.mpf(5)/2) * ctx.exp(-5*T) / 5
    total = low + high
    return {
        "value": stringify(total),
        "requested_dps": dps,
        "working_dps": dps + guard,
        "variant": variant,
        "backend": "mpmath " + mpmath.__version__,
        "method": "Direct K0 evaluation and tanh-sinh quadrature",
        "integral": "integral_0^infinity K0(t)^5 dt (no factor t)",
        "low_integral": stringify(low),
        "high_integral": stringify(high),
        "low_settings": low_settings,
        "upper_t_cutoff": stringify(T),
        "upper_segments": segment_details,
        "omitted_lower_tail_bound_estimate": stringify(lower_tail),
        "omitted_upper_tail_bound_estimate": stringify(upper_tail),
        "omitted_tail_upper_bound_estimate": stringify(lower_tail + upper_tail),
        "upper_tail_bound_formula": "(pi/(2*T))^(5/2)*exp(-5*T)/5",
        "quadrature_error_heuristic": stringify(abs(low_error) + high_error),
        "error_status":
            "Analytic tail inequalities evaluated in floating point; quadrature and roundoff are not rigorously enclosed. Check convergence across precision and substitutions.",
        "elapsed_seconds": round(time.perf_counter() - started, 6),
        "ai_disclosure": "All the work was done by AI.",
    }

## 4. Independently compute the modular side

Dedekind eta is evaluated from its Euler pentagonal expansion.
The transformation $\eta(ix)=\eta(i/x)/\sqrt{x}$ keeps the series rapidly convergent.
We integrate the defining Mellin transform, changing to $u=\log y$ in the alternate run.
This computation does not use the direct Bessel calculation or its answer.
Both routes share the `mpmath` arithmetic library, so they are not fully independent software implementations.


In [3]:
"""Fresh numerical Mellin integration for the Bessel 5 modular expression.

All the work was done by AI.

Only the eta-quotient mathematical definition in upstream/main.tex is used.
No archived numerical values, upstream routines, or coefficient tables are
imported. This is numerical convergence evidence, not a rigorous enclosure or
a proof of the Bessel conjecture.
"""

from __future__ import annotations

import json
import math
import time

import mpmath


def _eta_imaginary(mp, x, terms: int):
    """Dedekind eta(i*x), x>0, using its S transformation and pentagonal sum.

    eta(i*x) = exp(-pi*x/12) prod_{n>=1}(1-exp(-2*pi*n*x)).
    Euler's pentagonal theorem evaluates the product as
    1 + sum_{n>=1} (-1)^n (q^(n(3n-1)/2)+q^(n(3n+1)/2)).
    Modular transformation keeps q <= exp(-2*pi); all square roots are positive.
    """
    if x <= 0:
        raise ValueError("eta argument must be positive")
    z = x if x >= 1 else 1 / x
    q = mp.exp(-2 * mp.pi * z)
    product = mp.mpf(1)
    for n in range(1, terms + 1):
        pair = q ** (n * (3 * n - 1) // 2) + q ** (n * (3 * n + 1) // 2)
        product += pair if n % 2 == 0 else -pair
    eta = mp.exp(-mp.pi * z / 12) * product
    return eta if x >= 1 else eta / mp.sqrt(x)


def _embedding_check(mp, r, omega, epsilon):
    """Fresh integer Euler-product coefficients through q^7, not a data table."""
    degree = 7
    definitions = (
        (1, {2: 1, 6: 2, 10: 4, 30: -1}),
        (3, {2: 4, 6: -1, 10: 1, 30: 2}),
        (3, {2: -1, 6: 4, 10: 2, 30: 1}),
        (5, {2: 2, 6: 1, 10: -1, 30: 4}),
    )
    rows = []
    for shift, powers in definitions:
        row = [1] + [0] * degree
        for d, exponent in powers.items():
            for step in range(d, degree + 1, d):
                if exponent == -1:
                    # Multiplication by the exact formal geometric series.
                    for k in range(step, degree + 1):
                        row[k] += row[k-step]
                else:
                    for _ in range(exponent):
                        for k in range(degree, step-1, -1):
                            row[k] -= row[k-step]
        rows.append([0]*shift + row[:degree+1-shift])
    weights = (mp.mpc(1), omega, 3*epsilon, 3*epsilon*mp.conj(omega))
    expected = {1: mp.mpc(1), 3: r-2j, 5: r*(1+2j), 7: mp.mpc(0, -8)}
    errors = {}
    values = {}
    for n, target in expected.items():
        value = sum(weight*row[n] for weight, row in zip(weights, rows))
        error = abs(value-target)
        errors[str(n)] = mp.nstr(error, 8)
        values[str(n)] = {"real": mp.nstr(mp.re(value), 18), "imag": mp.nstr(mp.im(value), 18)}
        if error > mp.mpf(10)**(-mp.dps+5):
            raise ArithmeticError(f"Unexpected q^{n} coefficient / complex embedding")
    return {"passed": True, "method": "Fresh formal integer Euler products through q^7",
            "coefficients": values, "absolute_errors": errors}


def compute_lvalue(dps: int, variant: str = "primary") -> dict:
    """Return L(f,2) and the proposed Bessel expression as decimal strings.

    primary: integrate y*f(iy) directly on a partition of (0,infinity).
    shifted: integrate after y=exp(u), so the integrand is exp(2u)*f(i*exp(u)),
             on a differently partitioned finite interval, with rapidly
             decaying numerical tails truncated outside it.

    Values have dps significant digits; arithmetic uses 20 guard digits.
    Neither mpmath's adaptive integration nor the truncated tails supplies
    a rigorous interval certificate. Compare independent runs/partitions.
    """
    if not isinstance(dps, int) or not 20 <= dps <= 300:
        raise ValueError("dps must be an integer between 20 and 300")
    if variant not in ("primary", "shifted"):
        raise ValueError("variant must be 'primary' or 'shifted'")
    start = time.perf_counter()
    mp = mpmath.mp.clone()
    mp.dps = dps + 20
    terms = math.ceil(math.sqrt((mp.dps + 10) * math.log(10) / (3 * math.pi))) + 2
    r = mp.sqrt(5)
    omega = (1 - 2j) / r
    epsilon = (5 + 4 * r + 1j * (2 * r - 10)) / 15
    embedding_check = _embedding_check(mp, r, omega, epsilon)
    calls = 0

    def f(y):
        nonlocal calls
        calls += 1
        e2, e6, e10, e30 = [_eta_imaginary(mp, d * y, terms) for d in (2, 6, 10, 30)]
        p1 = e2 * e6**2 * e10**4 / e30
        p2 = e2**4 * e10 * e30**2 / e6
        p3 = e6**4 * e10**2 * e30 / e2
        p4 = e2**2 * e6 * e30**4 / e10
        return p1 + omega * p2 + 3 * epsilon * p3 + 3 * epsilon * mp.conj(omega) * p4

    if variant == "primary":
        partition = [mp.mpf(0), mp.mpf(1)/60, mp.mpf(1)/30, mp.mpf(1)/10,
                     mp.mpf(1)/6, mp.mpf(1)/2, mp.mpf(1), mp.mpf(3), mp.inf]
        value = 4 * mp.pi**2 * mp.quad(lambda y: y * f(y), partition)
        variable = "y; full positive real axis"
        tails = "No explicit tail cutoff; mpmath handles infinite endpoints numerically."
    else:
        # The slowest small-y eta quotient has exponential exp(-pi/(30*y))
        # times a power y^-3. At y=exp(-12), its exponential is below exp(-17000).
        # At y=exp(8), the large-y leading q term decays below exp(-18000).
        partition = [mp.mpf(k) for k in (-12, -6, -4, -3, -2, -1, 0, 1, 3, 8)]
        value = 4 * mp.pi**2 * mp.quad(lambda u: mp.exp(2*u) * f(mp.exp(u)), partition)
        variable = "u=log(y); y in [exp(-12),exp(8)]"
        tails = ("Numerical tails omitted outside [exp(-12),exp(8)]; eta asymptotic "
                 "decay is far below requested precision, but no interval bound is claimed.")
    square = value**2
    rhs = mp.pi / 4 * ((125 - 13*r) * mp.re(square) + (41*r - 125) * mp.im(square))
    # A conservative magnitude bound for the omitted pentagonal terms at z>=1:
    # 2*q^a/(1-q), a=(N+1)(3(N+1)-1)/2, since exponents increase by >=1.
    n = terms + 1
    qmax = mp.exp(-2 * mp.pi)
    omitted_product_bound = 2*qmax**(n*(3*n-1)//2)/(1-qmax)
    return {
        "real": mp.nstr(mp.re(value), dps),
        "imag": mp.nstr(mp.im(value), dps),
        "rhs": mp.nstr(rhs, dps),
        "settings": {
            "dps": dps, "working_dps": mp.dps, "variant": variant,
            "quadrature": "mpmath tanh-sinh", "variable": variable,
            "partition": [mp.nstr(x, 18) for x in partition],
            "eta_method": "S transformation and fresh Euler pentagonal sum",
            "eta_terms_each_side": terms,
            "eta_product_truncation_bound": mp.nstr(omitted_product_bound, 8),
            "tails": tails,
            "integrand_evaluations": calls,
            "mpmath_version": mpmath.__version__,
            "rigorous_enclosure": False,
            "source": "Mathematical eta-quotient definition in upstream/main.tex only",
            "embedding_check": embedding_check,
        },
        "seconds": round(time.perf_counter()-start, 6),
        "disclosure": "All the work was done by AI.",
    }

## 5. Run both calculations from scratch

Default: 40-digit primary and 60-digit alternate runs. For a longer check,
change `PRECISIONS` to `[40, 60, 80]` and rerun this cell and all cells below it.
Run each side completely before comparing; never use the proposed equality to obtain either input.


In [4]:
from datetime import datetime, timezone
import json, time
PRECISIONS = [40, 60]
if len(PRECISIONS) < 2 or PRECISIONS != sorted(set(PRECISIONS)) or min(PRECISIONS) < 30:
    raise ValueError("Choose at least two increasing precisions >=30")
runs = []
started_utc = datetime.now(timezone.utc).isoformat()
for index, digits in enumerate(PRECISIONS):
    print(f"Computing {digits} digits...", flush=True)
    bessel = compute_bessel(digits, "primary" if index % 2 == 0 else "alternate")
    modular = compute_lvalue(digits, "primary" if index % 2 == 0 else "shifted")
    runs.append({"dps": digits, "bessel": bessel, "modular": modular})
    print("Direct integral:", bessel["value"])
    print("Modular RHS:    ", modular["rhs"])
    print("L real:         ", modular["real"])
    print("L imaginary:    ", modular["imag"])
    print("Seconds:", round(bessel["elapsed_seconds"] + modular["seconds"], 3), "\n")

Computing 40 digits...


Direct integral: 135.268302580868837594226279646192207420305889359
Modular RHS:     135.2683025808688375942262796461922074203
L real:          1.291976871889574966527300485843216091404
L imaginary:     -0.1744414763789383543347225274297312549237
Seconds: 4.957 

Computing 60 digits...


Direct integral: 135.26830258086883759422627964619220742030588935942352678469351371046
Modular RHS:     135.268302580868837594226279646192207420305889359423526784694
L real:          1.29197687188957496652730048584321609140445653787241510901889
L imaginary:     -0.17444147637893835433472252742973125492367888876936754558164
Seconds: 12.8 



## 6. Convergence and deliberate failure controls

Before seeing the results, set the relative comparison threshold at
$10^{-(p-10)}$ for a run requesting $p$ significant digits.
For adjacent runs, test convergence at the threshold of the lower precision.
Decimal strings are compared with extra working precision, not ordinary Python floats.

Three controls deliberately use the wrong complex embedding, omit the Mellin factor,
or replace $L^2$ by $|L|^2$. Each must disagree by more than 0.1%.
A synthetic relative perturbation of $10^{-12}$ must also be detected.
These test sensitivity; they do not prove the original formula.


In [5]:
"""Compare freshly computed decimal strings without lowering precision.

All the work was done by AI. These are empirical tests, not interval proofs.
"""
import mpmath


def assess_checks(runs):
    """Apply thresholds fixed in advance: requested digits minus ten.

    Each run compares separate computations. Adjacent runs test sensitivity to
    precision and coordinates. Negative controls use an incorrect phase and an
    omitted Mellin normalization; a perturbed result tests the comparison itself.
    """
    if len(runs) < 2:
        raise ValueError("At least two precision/coordinate runs are required")
    precisions = [run["dps"] for run in runs]
    if precisions != sorted(set(precisions)) or min(precisions) < 30:
        raise ValueError("Use increasing, distinct precisions >=30")
    mp = mpmath.mp.clone()
    mp.dps = max(run["dps"] for run in runs) + 30
    rows = []
    previous = None
    for run in runs:
        dps = run["dps"]
        a = mp.mpf(run["bessel"]["value"])
        rhs = mp.mpf(run["modular"]["rhs"])
        tolerance = mp.mpf(10) ** (-(dps - 10))
        error = abs(a-rhs)/abs(a)
        row = {"dps": dps, "relative_discrepancy": mp.nstr(error, 12),
               "threshold": mp.nstr(tolerance, 8), "passed": bool(error < tolerance)}
        if previous is not None:
            low, old_a, old_rhs = previous
            convergence_tolerance = mp.mpf(10) ** (-(low - 10))
            da, dr = abs(a-old_a)/abs(a), abs(rhs-old_rhs)/abs(rhs)
            row["convergence"] = {
                "previous_dps": low, "bessel_relative_change": mp.nstr(da, 12),
                "rhs_relative_change": mp.nstr(dr, 12),
                "threshold": mp.nstr(convergence_tolerance, 8),
                "passed": bool(max(da, dr) < convergence_tolerance)}
        rows.append(row)
        previous = (dps, a, rhs)
    final = runs[-1]
    L = mp.mpc(final["modular"]["real"], final["modular"]["imag"])
    r = mp.sqrt(5)
    def expression(z):
        return mp.pi/4*((125-13*r)*mp.re(z**2)+(41*r-125)*mp.im(z**2))
    controls = {
        "conjugate_complex_embedding": expression(mp.conj(L)),
        "omit_mellin_4pi_squared": expression(L/(4*mp.pi**2)),
        "replace_L_squared_by_abs_L_squared": mp.pi/4*(125-13*r)*abs(L)**2,
    }
    negative = {}
    for name, value in controls.items():
        error = abs(a-value)/abs(a)
        negative[name] = {"value": mp.nstr(value, 24),
                          "relative_discrepancy": mp.nstr(error, 12),
                          "rejected": bool(error > mp.mpf("0.001"))}
    # Meaningful sensitivity test: injecting a 1e-12 relative change must fail.
    perturbation_detected = bool(abs(a*(1+mp.mpf("1e-12"))-rhs)/abs(a) > tolerance)
    passed = (all(x["passed"] and x.get("convergence", {}).get("passed", True) for x in rows)
              and all(x["rejected"] for x in negative.values()) and perturbation_detected)
    return {"passed": passed, "comparisons": rows, "negative_controls": negative,
            "relative_1e_minus_12_perturbation_detected": perturbation_detected,
            "rigorous_enclosure": False,
            "interpretation": "Empirical numerical agreement; not a proof or human peer review.",
            "disclosure": "All the work was done by AI."}

In [6]:
assessment = assess_checks(runs)
print(json.dumps(assessment, indent=2))
if not assessment["passed"]:
    raise ArithmeticError("A convergence, agreement or negative-control check failed")
print("PASS: empirical agreement and convergence; incorrect controls rejected.")
print("The Bessel equality remains conjectural. All the work was done by AI.")

{
  "passed": true,
  "comparisons": [
    {
      "dps": 40,
      "relative_discrepancy": "4.35383522055e-41",
      "threshold": "1.0e-30",
      "passed": true
    },
    {
      "dps": 60,
      "relative_discrepancy": "3.5949999425e-60",
      "threshold": "1.0e-50",
      "passed": true,
      "convergence": {
        "previous_dps": 40,
        "bessel_relative_change": "3.13101278432e-48",
        "rhs_relative_change": "4.35383553365e-41",
        "threshold": "1.0e-30",
        "passed": true
      }
    }
  ],
  "negative_controls": {
    "conjugate_complex_embedding": {
      "value": "111.675737194513115718267",
      "relative_discrepancy": "0.174413110361",
      "rejected": true
    },
    "omit_mellin_4pi_squared": {
      "value": "0.0867913746197794216298119",
      "relative_discrepancy": "0.999358376109",
      "rejected": true
    },
    "replace_L_squared_by_abs_L_squared": {
      "value": "128.05743737029113998954",
      "relative_discrepancy": "0.05330787089

## 7. Preserve the complete fresh-run evidence

In [7]:
from pathlib import Path
report = {
    "started_utc": started_utc,
    "finished_utc": datetime.now(timezone.utc).isoformat(),
    "python": sys.version, "platform": platform.platform(),
    "mpmath": mpmath.__version__, "runs": runs, "assessment": assessment,
    "input_policy": "Definitions only; no upstream numerical data or saved answers read.",
    "disclosure": "All the work was done by AI."
}
output_path = Path("bessel5-fresh-" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ") + ".json")
output_path.write_text(json.dumps(report, indent=2) + "\n")
print("Saved", output_path)
print("Download this JSON and your executed notebook to retain the settings and results.")

Saved bessel5-fresh-20261016T164933076116Z.json
Download this JSON and your executed notebook to retain the settings and results.


## What this establishes—and what remains open

A passing run supplies fresh numerical evidence using direct Bessel quadrature and
the defining eta-Mellin integral, with changed precision and coordinates.
It does not provide a rigorous global error enclosure or a proof of equality.
The original certificate pipeline is a separate method, preserved unmodified in `upstream/`.
No inference of external peer review or human verification is intended.

**All the work was done by AI.** Jonas Matuzas retains responsibility for the claims.

References: [paper and source v4](https://doi.org/10.5281/zenodo.22793737),
[DLMF Bessel integral representations](https://dlmf.nist.gov/10.32),
[DLMF eta transformations](https://dlmf.nist.gov/23.18),
[mpmath integration documentation](https://mpmath.org/doc/current/calculus/integration.html).
